In [1]:
import pandas as pd

df = pd.read_json(
    "../data/processed/Movies_Features.json"
)

print(df.head())
print(df.shape)
print(df["vote_count"].describe())

   movie_id                      title  \
0    969681  Spider-Man: Brand New Day   
1   1423191              Resident Evil   
2   1101383      The End of Oak Street   
3   1204680            Coyote vs. Acme   
4   1368337                The Odyssey   

                                            overview  release_date  runtime  \
0  Fighting crime full-time as Spider-Man in a wo...  1.785283e+12      145   
1  Medical courier Bryan unwittingly finds himsel...  1.789517e+12       95   
2  After a mysterious cosmic event rips Oak Stree...  1.786493e+12      100   
3  After Acme products fail him one too many time...  1.786925e+12      103   
4  Odysseus, the legendary King of Ithaca, embark...  1.784074e+12      173   

  original_language                                genres  \
0                en  [Science Fiction, Action, Adventure]   
1                en  [Horror, Science Fiction, Adventure]   
2                en  [Science Fiction, Mystery, Thriller]   
3                en         

In [2]:
threshold = df["vote_count"].quantile(0.75)

print("Threshold:", threshold)

df["high_engagement"] = (
    df["vote_count"] >= threshold
).astype(int)

print(df["high_engagement"].value_counts())

Threshold: 6576.25
high_engagement
0    2055
1     685
Name: count, dtype: int64


In [3]:
features = [
    "runtime",
    "budget",
    "revenue",
    "popularity",
    "vote_average",
    "num_genres",
    "num_keywords",
    "release_year",
    "release_month",
    "release_decade",
    "profit"
]

X = df[features]
y = df["high_engagement"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2740, 11)
y shape: (2740,)


In [4]:
print(X.isnull().sum())
X = X.fillna(0)
X = df[features]

X = X.fillna(0)

y = df["high_engagement"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

runtime           0
budget            0
revenue           0
popularity        0
vote_average      0
num_genres        0
num_keywords      0
release_year      1
release_month     1
release_decade    1
profit            0
dtype: int64


In [5]:
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (2192, 11)
Testing: (548, 11)


In [6]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000
)

logistic_model.fit(
    X_train,
    y_train
)
y_pred_logistic = logistic_model.predict(
    X_test
)

In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(
    y_test,
    y_pred_logistic
)

precision = precision_score(
    y_test,
    y_pred_logistic
)

recall = recall_score(
    y_test,
    y_pred_logistic
)

f1 = f1_score(
    y_test,
    y_pred_logistic
)
y_prob_logistic = logistic_model.predict_proba(
    X_test
)[:, 1]

roc_auc = roc_auc_score(
    y_test,
    y_prob_logistic
)
print("Logistic Regression")
print("-------------------")
print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC-AUC  :", roc_auc)

cm = confusion_matrix(
    y_test,
    y_pred_logistic
)

print(cm)

Logistic Regression
-------------------
Accuracy : 0.8065693430656934
Precision: 0.7123287671232876
Recall   : 0.3795620437956204
F1 Score : 0.49523809523809526
ROC-AUC  : 0.820945885946685
[[390  21]
 [ 85  52]]


In [8]:
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

random_forest.fit(
    X_train,
    y_train
)

y_pred_rf = random_forest.predict(
    X_test
)

y_prob_rf = random_forest.predict_proba(
    X_test
)[:, 1]


accuracy_rf = accuracy_score(
    y_test,
    y_pred_rf
)

precision_rf = precision_score(
    y_test,
    y_pred_rf
)

recall_rf = recall_score(
    y_test,
    y_pred_rf
)

f1_rf = f1_score(
    y_test,
    y_pred_rf
)

roc_auc_rf = roc_auc_score(
    y_test,
    y_prob_rf
)

print("Random Forest")
print("-------------")
print("Accuracy :", accuracy_rf)
print("Precision:", precision_rf)
print("Recall   :", recall_rf)
print("F1 Score :", f1_rf)
print("ROC-AUC  :", roc_auc_rf)

Random Forest
-------------
Accuracy : 0.8686131386861314
Precision: 0.7927927927927928
Recall   : 0.6423357664233577
F1 Score : 0.7096774193548387
ROC-AUC  : 0.9281172855950415


In [9]:
from sklearn.svm import LinearSVC

svm_model = LinearSVC(
    random_state=42,
    max_iter=5000
)

svm_model.fit(
    X_train,
    y_train
)

y_pred_svm = svm_model.predict(
    X_test
)

accuracy_svm = accuracy_score(
    y_test,
    y_pred_svm
)

precision_svm = precision_score(
    y_test,
    y_pred_svm
)

recall_svm = recall_score(
    y_test,
    y_pred_svm
)

f1_svm = f1_score(
    y_test,
    y_pred_svm
)

y_score_svm = svm_model.decision_function(
    X_test
)

roc_auc_svm = roc_auc_score(
    y_test,
    y_score_svm
)

print("Linear SVM")
print("----------")
print("Accuracy :", accuracy_svm)
print("Precision:", precision_svm)
print("Recall   :", recall_svm)
print("F1 Score :", f1_svm)
print("ROC-AUC  :", roc_auc_svm)

Linear SVM
----------
Accuracy : 0.8065693430656934
Precision: 0.6781609195402298
Recall   : 0.4306569343065693
F1 Score : 0.5267857142857143
ROC-AUC  : 0.742643010638109


In [10]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest",
        "Linear SVM"
    ],
    "Accuracy": [
        accuracy,
        accuracy_rf,
        accuracy_svm
    ],
    "Precision": [
        precision,
        precision_rf,
        precision_svm
    ],
    "Recall": [
        recall,
        recall_rf,
        recall_svm
    ],
    "F1": [
        f1,
        f1_rf,
        f1_svm
    ],
    "ROC-AUC": [
        roc_auc,
        roc_auc_rf,
        roc_auc_svm
    ]
})

print(results)

                 Model  Accuracy  Precision    Recall        F1   ROC-AUC
0  Logistic Regression  0.806569   0.712329  0.379562  0.495238  0.820946
1        Random Forest  0.868613   0.792793  0.642336  0.709677  0.928117
2           Linear SVM  0.806569   0.678161  0.430657  0.526786  0.742643
